In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "data" / "loader.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\Projects\Factor-Research


In [3]:
import pandas as pd
from src.data.loader import PROJECT_ROOT
from factors.momentum import calculate_momentum
from src.evaluation import forward_returns

path = (
    PROJECT_ROOT
    / "data/market/2026-11-12/prices.csv"
)

prices = pd.read_csv(
    path,
    index_col=0,
    parse_dates=True
)

prices = prices.sort_index()

print(prices.shape)
print(prices.index.min(), prices.index.max())

(4024, 300)
2010-01-04 00:00:00 2025-12-31 00:00:00


In [4]:
momentum = calculate_momentum(
    prices,
    window=20
)

forward_5d = forward_returns(
    prices,
    horizon=5,
    entry_lag=1
)

In [5]:
valid_pairs = (
    momentum.notna()
    & forward_5d.notna()
)

n_assets = valid_pairs.sum(axis=1)

print(n_assets.describe())
print("Minimum:", n_assets.min())
print("Maximum:", n_assets.max())

count    4024.000000
mean      275.001491
std        26.156169
min         0.000000
25%       266.000000
50%       276.000000
75%       290.000000
max       298.000000
dtype: float64
Minimum: 0
Maximum: 298


In [6]:
from src.evaluation import rank_ic, ic_summary

# Exploration period only
factor_train = momentum.loc[
    "2010-01-01":"2016-12-31"
]

returns_train = forward_5d.loc[
    "2010-01-01":"2016-12-31"
]

# Daily cross-sectional Rank IC
daily_ic = rank_ic(
    factor_train,
    returns_train,
    min_assets=100
)

summary = ic_summary(daily_ic)

print(summary)

count                1742.000000
mean                   -0.010083
std                     0.157018
icir                   -0.064218
positive_fraction       0.465557
dtype: float64


In [7]:
train_end = pd.Timestamp("2016-12-31")

endpoint_dates = pd.Series(
    prices.index.to_series().shift(-6).to_numpy(),
    index=prices.index
)

daily_ic = daily_ic.where(
    endpoint_dates.reindex(daily_ic.index) <= train_end
)

print(ic_summary(daily_ic))

count                1736.000000
mean                   -0.009835
std                     0.157147
icir                   -0.062588
positive_fraction       0.466590
dtype: float64


In [9]:
yearly_ic = daily_ic.groupby(daily_ic.index.year).agg(
    ["count", "mean", "std"]
)

yearly_ic["positive_fraction"] = (
    daily_ic.dropna()
    .gt(0)
    .groupby(daily_ic.dropna().index.year)
    .mean()
)

print(yearly_ic)

      count      mean       std  positive_fraction
Date                                              
2010    232 -0.008834  0.128394           0.491379
2011    252 -0.008816  0.174686           0.444444
2012    250 -0.009322  0.142814           0.392000
2013    252  0.003346  0.143432           0.531746
2014    252 -0.017844  0.141916           0.464286
2015    252  0.018396  0.163887           0.563492
2016    246 -0.046566  0.188946           0.378049


In [10]:
ic_without_2016 = daily_ic[
    daily_ic.index.year < 2016
]

print(ic_summary(ic_without_2016))

count                1490.000000
mean                   -0.003771
std                     0.150471
icir                   -0.025063
positive_fraction       0.481208
dtype: float64


In [11]:
universe = pd.read_csv(
    PROJECT_ROOT / "config/universe.csv"
)

sector_map = universe.set_index("ticker")["sector"]

print(sector_map.value_counts())

sector
Industrials               50
Information Technology    45
Financials                45
Health Care               36
Consumer Discretionary    28
Consumer Staples          20
Utilities                 18
Real Estate               18
Materials                 14
Energy                    13
Communication Services    13
Name: count, dtype: int64


In [12]:
# Factor matrix: Date × Ticker
raw_momentum = momentum.copy()

sector_neutral = pd.DataFrame(
    index=raw_momentum.index,
    columns=raw_momentum.columns,
    dtype=float
)

for sector in sector_map.dropna().unique():

    tickers = [
        ticker for ticker in raw_momentum.columns
        if sector_map.get(ticker) == sector
    ]

    sector_values = raw_momentum[tickers]

    sector_neutral[tickers] = sector_values.sub(
        sector_values.mean(axis=1),
        axis=0
    )

In [13]:
from src.evaluation import rank_ic, ic_summary

sn_train = sector_neutral.loc[
    "2010-01-01":"2016-12-31"
]

returns_train = forward_5d.loc[
    "2010-01-01":"2016-12-31"
]

sn_ic = rank_ic(
    sn_train,
    returns_train,
    min_assets=100
)

sn_ic = sn_ic.where(
    endpoint_dates.reindex(sn_ic.index) <= train_end
)

print(ic_summary(sn_ic))

count                1736.000000
mean                   -0.012524
std                     0.112791
icir                   -0.111041
positive_fraction       0.459677
dtype: float64


In [14]:
comparison = pd.DataFrame({
    "Raw": daily_ic,
    "Sector Neutral": sn_ic
})

yearly_comparison = comparison.groupby(
    comparison.index.year
).mean()

print(yearly_comparison)

           Raw  Sector Neutral
Date                          
2010 -0.008834       -0.012691
2011 -0.008816       -0.010132
2012 -0.009322       -0.020140
2013  0.003346       -0.006940
2014 -0.017844       -0.012854
2015  0.018396        0.018471
2016 -0.046566       -0.044213


In [15]:
from src.evaluation import rank_ic, ic_summary

sector_results = {}

for sector in sector_map.dropna().unique():

    tickers = [
        t for t in momentum.columns
        if sector_map.get(t) == sector
    ]

    factor_sector = momentum.loc[
        "2010-01-01":"2016-12-31", tickers
    ]

    returns_sector = forward_5d.loc[
        "2010-01-01":"2016-12-31", tickers
    ]

    ic_sector = rank_ic(
        factor_sector,
        returns_sector,
        min_assets=5
    )

    ic_sector = ic_sector.where(
        endpoint_dates.reindex(ic_sector.index) <= train_end
    )

    sector_results[sector] = ic_summary(ic_sector)

sector_table = pd.DataFrame(sector_results).T

print(
    sector_table.sort_values("mean")
)

                         count      mean       std      icir  \
Real Estate             1736.0 -0.039759  0.320461 -0.124069   
Health Care             1736.0 -0.035142  0.221323 -0.158780   
Industrials             1736.0 -0.023283  0.201239 -0.115698   
Utilities               1736.0 -0.021324  0.298989 -0.071319   
Financials              1736.0 -0.015630  0.237664 -0.065764   
Consumer Staples        1736.0 -0.011849  0.263411 -0.044983   
Consumer Discretionary  1736.0 -0.008665  0.246186 -0.035197   
Materials               1736.0 -0.008173  0.337750 -0.024200   
Information Technology  1736.0 -0.006197  0.206569 -0.030000   
Communication Services  1736.0  0.014371  0.382535  0.037568   
Energy                  1736.0  0.028911  0.365664  0.079064   

                        positive_fraction  
Real Estate                      0.455069  
Health Care                      0.414747  
Industrials                      0.455645  
Utilities                        0.479263  
Financials 

## Exploratory appendix: SPY market-trend regimes (2010–2016 only)

This fixed-specification extension does not replace prior research results or conclusions.
Raw 20-day Momentum is unchanged. Target: P[t+6]/P[t+1]-1, with at least 100
paired assets per signal date, matching the existing daily Rank IC series.
Equity inputs are sliced to 2010–2016 before computing labels. Both endpoints
must fall inside this period. No 2017–2025 IC values are computed or used.
The initial equity factor warm-up remains unchanged (20 rows in 2010).

SPY adjusted daily Close is obtained through the existing validated loader for
2008-01-01 through 2017-01-01 exclusive, with raw and processed request caches.
Its trailing 200-row arithmetic moving average includes the signal-date close.
Bull means Close > SMA; Bear means Close <= SMA. Labels are known after that
close, consistent with next-row entry. The first 199 rows and windows containing
missing prices are unlabeled; no filling is used. Pre-2010 SPY data warms up
the SMA before labels are aligned by signal date, never by row position.

Included dates: 1736; excluded exploration rows: 26.
Standard deviation uses ddof=1. ICIR is mean/std, unannualized. Positive fraction
is the proportion of valid daily IC values strictly greater than zero.
Yearly counts below refer to included IC observations, not all market sessions.

### Conditional Rank IC

| regime | count | mean | std | icir | positive_fraction |
| --- | --- | --- | --- | --- | --- |
| Bull | 1474.000000 | -0.007281 | 0.146984 | -0.049534 | 0.469471 |
| Bear | 262.000000 | -0.024208 | 0.204916 | -0.118138 | 0.450382 |

### Yearly distribution of included observations

| year | Bull | Bear | total | bull_fraction |
| --- | --- | --- | --- | --- |
| 2010 | 172 | 60 | 232 | 0.741379 |
| 2011 | 160 | 92 | 252 | 0.634921 |
| 2012 | 247 | 3 | 250 | 0.988000 |
| 2013 | 252 | 0 | 252 | 1.000000 |
| 2014 | 247 | 5 | 252 | 0.980159 |
| 2015 | 198 | 54 | 252 | 0.785714 |
| 2016 | 198 | 48 | 246 | 0.804878 |

### Descriptive visualization

![Conditional Rank IC](../reports/figures/momentum_001_regime_2010_2016.png)

### Scope and limitations

These are exploratory conditional descriptions, not a new research conclusion,
an executable strategy, or evidence of statistical significance. Five-day labels
overlap; IC observations and market regimes are serially dependent. No iid tests,
confidence intervals, or significance claims are made. Dependence-aware inference
would be necessary before assessing uncertainty. State counts are unequal.
The 200-day rule was specified in advance here; no thresholds or parameters were
tuned. Current-universe survivorship/selection bias remains. Yahoo adjusted prices
are retrieval-time snapshots subject to revisions and corporate-action adjustments,
not a certified point-in-time archive. Computational labeling uses only rows at or
before the signal date; this cannot remove vendor revision bias.

Structured outputs: `data/research/momentum_001/regime_2010_2016/` contains
daily alignment/inclusion flags, conditional statistics, yearly counts, SPY prices,
SMA/regimes, and methodology metadata. Reproduce with
`python -B scripts/momentum_regime_001.py`; in the notebook pass its existing
endpoint-filtered `daily_ic` to `run(daily_rank_ic=daily_ic)` for alignment verification.


In [ ]:
from scripts.momentum_regime_001 import run as run_regime_appendix

# Use the existing endpoint-filtered raw daily_ic (cell 5), not sector-neutral IC.
# This verifies equivalence to a reproduction bounded to 2010?2016.
regime_audit, regime_summary, regime_yearly, _ = run_regime_appendix(
    daily_rank_ic=daily_ic
)
print(regime_summary)
print(regime_yearly)

from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / "reports/figures/momentum_001_regime_2010_2016.png")))
